In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from pydrake.all import (
    DiagramBuilder,
    DiscreteTimeTrajectory,
    QuasidynamicLinearPusher,
    Meshcat,
    Simulator,
    TrajectorySource,
    VectorLogSink,
    InitializeAutoDiffTuple,
    ExtractValue,
    ExtractGradient,
)
from algorithms import (
    IterativeLQRController,
    IterativeSLSController,
    IterativeTO,
    TrajOptProblem,
)
IS_TEST = "BAZEL_TEST" in os.environ

In [ ]:
meshcat = Meshcat()

In [ ]:
k_a, m_o, w, dt = 1.0, 0.01, 0.1, 0.02

model = QuasidynamicLinearPusher(k_a=k_a, m_o=m_o, w=w, dt=dt, dynamics_smoothing=1e-3)
num_steps = 200 if not IS_TEST else 50

prob = TrajOptProblem(system=model, num_steps=num_steps)
x = prob.state()
u = prob.input()

prob.AddStageCost((x[1] - 0.5)**2 + 0.1 * u[0]**2)
prob.AddTerminalCost((x[1] - 0.5)**2)

x0 = [0.0, 0.2]
prob.SetInitialState(x0)

us_init = np.ones((num_steps, 1)) * 0.0
controller = IterativeLQRController(prob, us_init=us_init)

builder = DiagramBuilder()
linear_pusher = QuasidynamicLinearPusher(k_a=k_a, m_o=m_o, w=w, dt=dt)
linear_pusher.SetMeshcat(meshcat)
logger = VectorLogSink(2)
builder.AddSystem(linear_pusher)
builder.AddSystem(controller)
builder.AddSystem(logger)
builder.Connect(linear_pusher.get_state_output_port(), controller.get_input_port())
builder.Connect(controller.get_output_port(), linear_pusher.get_actuation_input_port())
builder.Connect(linear_pusher.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)
linear_pusher.SetState(
    diagram.GetMutableSubsystemContext(linear_pusher, simulator.get_context()),
    x0,
)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

xs = controller.xs
plt.plot(np.arange(len(xs)) * dt, xs[:, 1], label="Nominal trajectory")

log = logger.FindLog(simulator.get_context())
plt.plot(log.sample_times(), log.data()[1], label="Feedback control")

plt.title("IterativeLQR")
plt.axhline(y=0.5, color="k", linestyle="--", linewidth=1.0)
plt.xlabel("$t$")
plt.ylabel("$x_o$")
plt.legend(loc="lower right")
plt.xlim([0, num_steps * dt])
plt.ylim(bottom=0.2)
plt.show()

In [ ]:
for k in range(num_steps - 1):
    prob.AddCost((prob.input(k) - prob.input(k + 1))[0]**2)

prob.AddConstraint(prob.input(0), lb=[0.0], ub=[0.0])

traj = IterativeTO(
    prob,
    us_init=us_init,
    trust_region_covar=np.identity(3) * 0.1**-2,
    use_contact_trust_region=True,
)
us = traj.us
xs = traj.xs
u_traj = DiscreteTimeTrajectory(np.arange(len(us)) * dt, us[:, :, np.newaxis])

u_traj_source = TrajectorySource(u_traj)

builder = DiagramBuilder()
linear_pusher = QuasidynamicLinearPusher(k_a=k_a, m_o=m_o, w=w, dt=dt)
linear_pusher.SetMeshcat(meshcat)
logger = VectorLogSink(2)
builder.AddSystem(linear_pusher)
builder.AddSystem(u_traj_source)
builder.AddSystem(logger)
builder.Connect(u_traj_source.get_output_port(), linear_pusher.get_actuation_input_port())
builder.Connect(linear_pusher.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)
linear_pusher.SetState(
    diagram.GetMutableSubsystemContext(linear_pusher, simulator.get_context()),
    x0,
)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

plt.plot(np.arange(len(xs)) * dt, xs[:, 1], label="Nominal trajectory")

log = logger.FindLog(simulator.get_context())
plt.plot(log.sample_times(), log.data()[1], label="True dynamics rollout")

plt.title("IterativeTO with CTR")
plt.axhline(y=0.5, color="k", linestyle="--", linewidth=1.0)
plt.xlabel("$t$")
plt.ylabel("$x_o$")
plt.legend(loc="lower right")
plt.xlim([0, num_steps * dt])
plt.ylim(bottom=0.2)
plt.show()

In [ ]:
Q_bar=np.identity(2) * 1e2
R_bar=np.identity(1) * 1e2
Qf_bar=np.identity(2) * 1e3
prob.SetUncertaintyTubeCost(Q_bar=Q_bar, R_bar=R_bar, Qf_bar=Qf_bar)

Gf = np.array([[0.0, 1.0], [0.0, -1.0]])
gf = np.array([-0.5, 0.5])
prob.AddTerminalLinearConstraint(Gf=Gf, gf=gf)

controller = IterativeSLSController(
    prob,
    us_init=us_init,
    delta_cap=0.05,
    slack_penalty=1,
    max_outer_iters=18,
)

builder = DiagramBuilder()
linear_pusher = QuasidynamicLinearPusher(k_a=k_a, m_o=m_o, w=w, dt=dt)
linear_pusher.SetMeshcat(meshcat)
logger = VectorLogSink(2)
builder.AddSystem(linear_pusher)
builder.AddSystem(controller)
builder.AddSystem(logger)
builder.Connect(linear_pusher.get_state_output_port(), controller.get_input_port())
builder.Connect(controller.get_output_port(), linear_pusher.get_actuation_input_port())
builder.Connect(linear_pusher.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)
linear_pusher.SetState(
    diagram.GetMutableSubsystemContext(linear_pusher, simulator.get_context()),
    x0,
)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

dof = 1
ts = np.arange(num_steps + 1) * dt
plt.plot(ts, controller.xs[:, dof], label="Nominal trajectory")
plt.fill_between(ts, controller.xs_lb[:, dof], controller.xs_ub[:, dof], alpha=0.3)

log = logger.FindLog(simulator.get_context())
plt.plot(log.sample_times(), log.data()[1], label="Feedback control")

plt.title("IterativeSLS")
plt.axhline(y=0.5, color="k", linestyle="--", linewidth=1.0)
plt.xlabel("$t$")
plt.ylabel("$x_o$")
plt.legend(loc="lower right")
plt.xlim([0, num_steps * dt])
plt.ylim(bottom=0.2)
plt.show()

In [ ]:
k_a, m_o, w, dt = 1.0, 0.01, 0.1, 0.02

model = QuasidynamicLinearPusher(k_a=k_a, m_o=m_o, w=w, dt=dt).ToAutoDiffXd()
context = model.CreateDefaultContext()

def Dynamics(x, u, kappa):
    model.set_dynamics_smoothing(context, kappa)
    x_ad, u_ad = InitializeAutoDiffTuple(x, u)
    model.SetState(context, x_ad)
    model.get_actuation_input_port().FixValue(context, u_ad)
    context.SetDiscreteState(model.EvalUniquePeriodicDiscreteUpdate(context))
    xnext_ad = model.get_state_output_port().Eval(context)
    E_ad = model.GetDisturbanceMatrix(context)
    xnext = ExtractValue(xnext_ad).reshape(-1)
    E = ExtractValue(E_ad)
    return xnext, E, ExtractGradient(xnext_ad), ExtractGradient(E_ad)

U = np.linspace(-0.1, 0.3, num=101)
Xo_true = np.empty_like(U)
Xo = np.empty_like(U)
Eo = np.empty_like(U)
for i, u in enumerate(U):
    x = [0.0, 0.2]
    xnext, E, _, _ = Dynamics(x, [u], kappa=0.0)
    Xo_true[i] = xnext[1]

    xnext, E, _, _ = Dynamics(x, [u], kappa=1e-3)
    Xo[i] = xnext[1]
    Eo[i] = E[1].item()

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
plt.plot(U, Xo_true, label="True dynamics")
plt.plot(U, Xo, label="Smoothed dynamics")
plt.fill_between(
    U,
    Xo + Eo,
    Xo + Eo * 2,
    alpha=0.3,
    color=colors[1],
    edgecolor=None,
    label="Smoothed dynamics + correction",
)

u = 0.125
xnext, E, grad_xnext, grad_E = Dynamics(x, [u], kappa=1e-3)
E = E.reshape(-1)
grad_xnext = grad_xnext[:, -1]
grad_E = grad_E[:, -1]
du = 0.042
U = np.array([u - du, u + du])
Xo = np.array([(xnext - grad_xnext * du)[1], (xnext + grad_xnext * du)[1]])
Eo = np.array([(E - grad_E * du)[1], (E + grad_E * du)[1]])
plt.plot(U, Xo, label="Smoothed dynamics linearized")
plt.fill_between(
    U,
    Xo + Eo,
    Xo + Eo * 2,
    alpha=0.3,
    color=colors[4],
    edgecolor=None,
    label="Smoothed dynamics linearized + correction",
)

plt.xlabel("$u$")
plt.ylabel("$x_o^+$")
plt.xlim([-0.1, 0.3])
plt.legend(loc="upper left")
plt.show()